In [ ]:
# ============================================================
# COMP9417 Air Quality Project
# STEP 7 – REGRESSION MODELLING (Multi-Pollutant, Multi-Horizon)
# ============================================================
# Models used:
#   - Linear Regression
#   - XGBoost Regressor
#   - LSTM

# Predicts 5 pollutants:
#   CO(GT), NMHC(GT), C6H6(GT), NOx(GT), NO2(GT)
#
# Forecast horizons:
#   1h, 6h, 12h, 24h ahead
# ============================================================


import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("./AirQualityUCI.csv", sep=";")
print(df.head())


         Date      Time CO(GT)  PT08.S1(CO)  NMHC(GT) C6H6(GT)  PT08.S2(NMHC)  \
0  10/03/2004  18.00.00    2,6       1360.0     150.0     11,9         1046.0   
1  10/03/2004  19.00.00      2       1292.0     112.0      9,4          955.0   
2  10/03/2004  20.00.00    2,2       1402.0      88.0      9,0          939.0   
3  10/03/2004  21.00.00    2,2       1376.0      80.0      9,2          948.0   
4  10/03/2004  22.00.00    1,6       1272.0      51.0      6,5          836.0   

   NOx(GT)  PT08.S3(NOx)  NO2(GT)  PT08.S4(NO2)  PT08.S5(O3)     T    RH  \
0    166.0        1056.0    113.0        1692.0       1268.0  13,6  48,9   
1    103.0        1174.0     92.0        1559.0        972.0  13,3  47,7   
2    131.0        1140.0    114.0        1555.0       1074.0  11,9  54,0   
3    172.0        1092.0    122.0        1584.0       1203.0  11,0  60,0   
4    131.0        1205.0    116.0        1490.0       1110.0  11,2  59,6   

       AH  Unnamed: 15  Unnamed: 16  
0  0,7578         

In [ ]:
df = df.replace(",", ".", regex=True)

# Convert all numeric columns to float
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

# Replace -200 with NaN
df = df.replace(-200, np.nan)

print(df.isna().mean().sort_values(ascending=False))


Unnamed: 16      1.000000
Unnamed: 15      1.000000
NMHC(GT)         0.903495
CO(GT)           0.189737
NO2(GT)          0.185408
NOx(GT)          0.185091
PT08.S5(O3)      0.050681
AH               0.050681
RH               0.050681
T                0.050681
PT08.S3(NOx)     0.050681
PT08.S4(NO2)     0.050681
PT08.S2(NMHC)    0.050681
C6H6(GT)         0.050681
PT08.S1(CO)      0.050681
Time             0.012037
Date             0.012037
dtype: float64


In [ ]:
# --- FIX TIME FORMAT (dot → colon) ---
df['Time'] = df['Time'].astype(str).str.replace('.', ':', regex=False)

# --- CREATE TIMESTAMP ---
df['timestamp'] = pd.to_datetime(
    df['Date'] + " " + df['Time'], 
    dayfirst=True,
    errors='coerce'
)

# --- DROP BAD ROWS ---
df = df.dropna(subset=['timestamp'])
df = df.sort_values('timestamp')
df = df.set_index('timestamp')

# Remove old columns
df = df.drop(columns=['Date', 'Time'])


In [ ]:
df["hour"] = df.index.hour
df["weekday"] = df.index.weekday
df["month"] = df.index.month


In [ ]:
TARGETS = ['CO(GT)', 'NMHC(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)']

for col in TARGETS:
    df[f"{col}_lag1"] = df[col].shift(1)
    df[f"{col}_lag6"] = df[col].shift(6)
    df[f"{col}_lag12"] = df[col].shift(12)
    df[f"{col}_lag24"] = df[col].shift(24)
    df[f"{col}_roll24"] = df[col].rolling(24).mean()
    
horizons = [1, 6, 12, 24]  # hours ahead

for h in horizons:
    df[f"CO_t+{h}"] = df["CO(GT)"].shift(-h)
    df[f"NO2_t+{h}"] = df["NO2(GT)"].shift(-h)
    df[f"NOx_t+{h}"] = df["NOx(GT)"].shift(-h)
    df[f"C6H6_t+{h}"] = df["C6H6(GT)"].shift(-h)
    df[f"NMHC_t+{h}"] = df["NMHC(GT)"].shift(-h)
    
df = df.ffill().bfill()
df = df.drop(columns=["Unnamed: 15", "Unnamed: 16"], errors='ignore')


In [ ]:
train = df[:'2004-12-31']
test  = df['2005-01-01':]

val_size = int(len(train) * 0.1)
val = train[-val_size:]
train = train[:-val_size]

print("Train:", train.shape)
print("Val:", val.shape)
print("Test:", test.shape)


Train: (6399, 61)
Val: (711, 61)
Test: (2247, 61)


In [ ]:
scaler = StandardScaler()

train_scaled = pd.DataFrame(scaler.fit_transform(train), 
                            index=train.index, columns=train.columns)

val_scaled = pd.DataFrame(scaler.transform(val),
                          index=val.index, columns=val.columns)

test_scaled = pd.DataFrame(scaler.transform(test),
                           index=test.index, columns=test.columns)


In [ ]:
pollutants = ["CO", "NO2", "NOx", "C6H6", "NMHC"]
horizons = [1, 6, 12, 24]


In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

pollutant_map = {
    "CO(GT)": "CO",
    "NO2(GT)": "NO2",
    "NOx(GT)": "NOx",
    "C6H6(GT)": "C6H6",
    "NMHC(GT)": "NMHC"
}

horizons = [1, 6, 12, 24]

for original, short in pollutant_map.items():
    print(f"------ BASELINE RMSE FOR {original} ------")

    for h in horizons:
        true = test[f"{short}_t+{h}"]
        pred = test[original].shift(h)   # baseline uses actual column

        # safe alignment
        temp = pd.concat([true, pred], axis=1).dropna()
        y_true = temp.iloc[:, 0]
        y_pred = temp.iloc[:, 1]

        print(f"t+{h:2d} → RMSE = {rmse(y_true, y_pred):.4f}")


------ BASELINE RMSE FOR CO(GT) ------
t+ 1 → RMSE = 1.2165
t+ 6 → RMSE = 1.7659
t+12 → RMSE = 1.2522
t+24 → RMSE = 1.5098
------ BASELINE RMSE FOR NO2(GT) ------
t+ 1 → RMSE = 40.3834
t+ 6 → RMSE = 68.2782
t+12 → RMSE = 44.4534
t+24 → RMSE = 53.8278
------ BASELINE RMSE FOR NOx(GT) ------
t+ 1 → RMSE = 172.0393
t+ 6 → RMSE = 267.0622
t+12 → RMSE = 200.4637
t+24 → RMSE = 241.8474
------ BASELINE RMSE FOR C6H6(GT) ------
t+ 1 → RMSE = 5.5384
t+ 6 → RMSE = 8.3665
t+12 → RMSE = 5.9265
t+24 → RMSE = 7.2025
------ BASELINE RMSE FOR NMHC(GT) ------
t+ 1 → RMSE = 0.0000
t+ 6 → RMSE = 0.0000
t+12 → RMSE = 0.0000
t+24 → RMSE = 0.0000


In [58]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

pollutant_map = {
    "CO(GT)": "CO",
    "NO2(GT)": "NO2",
    "NOx(GT)": "NOx",
    "C6H6(GT)": "C6H6",
    "NMHC(GT)": "NMHC"
}

horizons = [1, 6, 12, 24]

# feature columns = all columns without future targets
feature_cols = [c for c in train.columns if "_t+" not in c]

for original, short in pollutant_map.items():
    print(f"------ LINEAR REGRESSION BASELINE FOR: {original} ------")

    for h in horizons:

        # y: target future value
        y_train = train[f"{short}_t+{h}"]
        y_test = test[f"{short}_t+{h}"]

        # X: features (scaled)
        X_train = train_scaled[feature_cols]
        X_test = test_scaled[feature_cols]

        # Train model
        model = LinearRegression()
        model.fit(X_train, y_train)

        # Predict
        pred = model.predict(X_test)

        # Evaluate
        error = rmse(y_test, pred)
        print(f"t+{h:2d} → RMSE = {error:.4f}")


------ LINEAR REGRESSION BASELINE FOR: CO(GT) ------
t+ 1 → RMSE = 0.6543
t+ 6 → RMSE = 1.2514
t+12 → RMSE = 1.1004
t+24 → RMSE = 1.0977
------ LINEAR REGRESSION BASELINE FOR: NO2(GT) ------
t+ 1 → RMSE = 24.0782
t+ 6 → RMSE = 53.7943
t+12 → RMSE = 45.0511
t+24 → RMSE = 46.1896
------ LINEAR REGRESSION BASELINE FOR: NOx(GT) ------
t+ 1 → RMSE = 99.5553
t+ 6 → RMSE = 218.2561
t+12 → RMSE = 203.2089
t+24 → RMSE = 191.3195
------ LINEAR REGRESSION BASELINE FOR: C6H6(GT) ------
t+ 1 → RMSE = 3.2039
t+ 6 → RMSE = 5.5833
t+12 → RMSE = 5.1108
t+24 → RMSE = 5.2828
------ LINEAR REGRESSION BASELINE FOR: NMHC(GT) ------
t+ 1 → RMSE = 9.6239
t+ 6 → RMSE = 19.2884
t+12 → RMSE = 18.4324
t+24 → RMSE = 29.9196


In [59]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import numpy as np

horizons = [1, 6, 12, 24]

# mapping between original columns and target shift names
pollutant_map = {
    "CO(GT)": "CO",
    "NO2(GT)": "NO2",
    "NOx(GT)": "NOx",
    "C6H6(GT)": "C6H6",
    "NMHC(GT)": "NMHC"
}

# Select feature columns (exclude all future targets)
feature_cols = [c for c in train.columns if "_t+" not in c]

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# TRAIN XGBOOST FOR ALL POLLUTANTS AND HORIZONS
results = {}

for original, short in pollutant_map.items():
    print(f"------ XGBOOST RESULTS FOR: {original} ------")
    results[original] = {}

    for h in horizons:

        # targets
        y_train = train[f"{short}_t+{h}"]
        y_test = test[f"{short}_t+{h}"]

        # features (already scaled)
        X_train = train_scaled[feature_cols]
        X_test = test_scaled[feature_cols]

        # Train XGBoost regressor
        model = xgb.XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='reg:squarederror',
            random_state=42
        )

        model.fit(X_train, y_train)

        # Predictions
        pred = model.predict(X_test)

        # RMSE
        error = rmse(y_test, pred)

        results[original][f"t+{h}"] = error
        print(f"t+{h:2d} ➜ RMSE = {error:.4f}")


------ XGBOOST RESULTS FOR: CO(GT) ------
t+ 1 ➜ RMSE = 0.6146
t+ 6 ➜ RMSE = 1.0272
t+12 ➜ RMSE = 1.2217
t+24 ➜ RMSE = 1.1872
------ XGBOOST RESULTS FOR: NO2(GT) ------
t+ 1 ➜ RMSE = 24.1088
t+ 6 ➜ RMSE = 40.5652
t+12 ➜ RMSE = 38.9030
t+24 ➜ RMSE = 41.2358
------ XGBOOST RESULTS FOR: NOx(GT) ------
t+ 1 ➜ RMSE = 100.7726
t+ 6 ➜ RMSE = 205.0583
t+12 ➜ RMSE = 177.1999
t+24 ➜ RMSE = 179.5573
------ XGBOOST RESULTS FOR: C6H6(GT) ------
t+ 1 ➜ RMSE = 3.0515
t+ 6 ➜ RMSE = 5.4137
t+12 ➜ RMSE = 6.5313
t+24 ➜ RMSE = 6.8014
------ XGBOOST RESULTS FOR: NMHC(GT) ------
t+ 1 ➜ RMSE = 13.7679
t+ 6 ➜ RMSE = 42.2421
t+12 ➜ RMSE = 8.5080
t+24 ➜ RMSE = 15.0755


In [60]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import mean_squared_error

def create_sequences(X, y, window=24):
    Xs, ys = [], []
    for i in range(len(X) - window):
        Xs.append(X[i:i+window])
        ys.append(y[i+window])
    return np.array(Xs), np.array(ys)

def build_lstm(input_dim, window=24):
    model = models.Sequential([
        layers.LSTM(64, return_sequences=True, input_shape=(window, input_dim)),
        layers.LSTM(32),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

pollutant_map = {
    "CO(GT)": "CO",
    "NO2(GT)": "NO2",
    "NOx(GT)": "NOx",
    "C6H6(GT)": "C6H6",
    "NMHC(GT)": "NMHC"
}

horizons = [1, 6, 12, 24]

window = 24
results_lstm = {}

feature_cols = [c for c in train.columns if "_t+" not in c]
input_dim = len(feature_cols)

for original, short in pollutant_map.items():
    print(f"------ LSTM RESULTS FOR: {original} ------")

    results_lstm[original] = {}

    for h in horizons:

        # Target vectors
        y_train = train[f"{short}_t+{h}"].values
        y_val   = val[f"{short}_t+{h}"].values
        y_test  = test[f"{short}_t+{h}"].values

        # Feature matrices
        X_train = train_scaled[feature_cols].values
        X_val   = val_scaled[feature_cols].values
        X_test  = test_scaled[feature_cols].values

        # Create sequences
        X_train_seq, y_train_seq = create_sequences(X_train, y_train, window)
        X_val_seq,   y_val_seq   = create_sequences(X_val, y_val, window)
        X_test_seq,  y_test_seq  = create_sequences(X_test, y_test, window)

        # Build model
        model = build_lstm(input_dim=input_dim, window=window)

        # Train model
        history = model.fit(
            X_train_seq, y_train_seq,
            validation_data=(X_val_seq, y_val_seq),
            epochs=10,
            batch_size=32,
            verbose=0
        )

        # Predict
        pred = model.predict(X_test_seq).flatten()

        # Evaluate
        error = rmse(y_test_seq, pred)
        results_lstm[original][f"t+{h}"] = error
        print(f"t+{h:2d} → RMSE = {error:.4f}")


ModuleNotFoundError: No module named 'tensorflow'